In [1]:
pip install -q faiss-cpu sentence-transformers rank-bm25 bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.4 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 73.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERRO

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful")

Hugging Face login successful


In [1]:
# ==========================================================
# INSTALL GPU LLAMA-CPP (KAGGLE)
# ==========================================================

# !pip uninstall -y llama-cpp-python -q use if cpp is installed


# Enable CUDA build
%env CMAKE_ARGS=-DGGML_CUDA=on
%env FORCE_CMAKE=1

!pip install llama-cpp-python -U \
    --force-reinstall \
    --no-cache-dir \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

env: CMAKE_ARGS=-DGGML_CUDA=on
env: FORCE_CMAKE=1
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 190.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 144.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 155.2 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.3
 

In [4]:
#cleaning llm from gpu

import gc
import torch

try:
    del llm
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1e9)
print(torch.cuda.memory_reserved() / 1e9)

0.0
0.0


In [1]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-31B-it-GGUF",
    filename="gemma-4-31B-it-Q3_K_S.gguf",

    n_gpu_layers=-1,
    split_mode=1,
    main_gpu=0,

    n_ctx=10000,
    n_batch=512,
    flash_attn=True,
    verbose=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gemma-4-31B-it-Q3_K_S.gguf:   0%|          | 0.00/13.2G [00:00<?, ?B/s]

llama_context: n_ctx_seq (10240) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


# Post Processing sub 16 (sub 16.1 creation)

In [2]:
import pandas as pd

# ==========================
# File paths
# ==========================
SUB16_PATH = "/kaggle/input/datasets/rathiaich/sub-16/submission16.csv"
SUB9_PATH = "/kaggle/input/datasets/rathiaich/sub-9-11/submission9.csv"

OUTPUT_PATH = "/kaggle/working/submission_16(1).csv"

# ==========================
# Load CSVs
# ==========================
sub16 = pd.read_csv(SUB16_PATH, encoding="utf-8")
sub9 = pd.read_csv(SUB9_PATH, encoding="utf-8")

# ==========================
# Find unanswered/problematic rows in submission16
# ==========================
regex = r"^\s*$|তথ্য\s+নেই|উল্লেখ\s+নেই|Context-এ\s+উল্লেখ\s+নেই|প্রদত্ত\s+তথ্যে"

mask = sub16["answer"].astype(str).str.contains(
    regex,
    regex=True,
    na=False
)

problem_rows = sub16.loc[mask, "index"]

print(f"Found {len(problem_rows)} problematic rows.")

# ==========================
# Create mapping from submission9
# ==========================
sub9_answer_map = dict(zip(sub9["index"], sub9["answer"]))

# ==========================
# Replace answers and print
# ==========================
replaced_count = 0
replacement_log = []

for idx in problem_rows:
    if idx in sub9_answer_map:

        # old answer from submission16
        old_answer = sub16.loc[
            sub16["index"] == idx, "answer"
        ].values[0]

        # new answer from submission9
        new_answer = sub9_answer_map[idx]

        # replace
        sub16.loc[
            sub16["index"] == idx, "answer"
        ] = new_answer

        replaced_count += 1

        # store log
        replacement_log.append({
            "index": idx,
            "old_answer": old_answer,
            "new_answer": new_answer
        })

print(f"\nReplaced {replaced_count} answers from submission9.\n")

# ==========================
# Print all replacements
# ==========================
for row in replacement_log:
    print("=" * 80)
    print(f"QUESTION ID: {row['index']}")
    print(f"OLD ANSWER: {row['old_answer']}")
    print(f"NEW ANSWER: {row['new_answer']}")

# ==========================
# Save merged submission
# ==========================
sub16.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nSaved file to:", OUTPUT_PATH)

Found 11 problematic rows.

Replaced 11 answers from submission9.

QUESTION ID: test_0350
OLD ANSWER: Context-এ উল্লেখ নেই
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_0481
OLD ANSWER: উল্লেখ নেই
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_0503
OLD ANSWER: তথ্য নেই
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_0634
OLD ANSWER: তথ্য নেই
NEW ANSWER: ২০২১ সালে
QUESTION ID: test_0692
OLD ANSWER: প্রদত্ত তথ্যে শ্যানজিয়াং মসজিদের মিনার ব্যবহারের কথা উল্লেখ নেই।
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_0697
OLD ANSWER: প্রদত্ত তথ্যে পৌরাণিকা বইটির প্রকাশের সাল উল্লেখ নেই।
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_0817
OLD ANSWER: প্রদত্ত তথ্যে নাজিয়াইং মসজিদটি কোন সালে সংরক্ষিত সাংস্কৃতিক নিদর্শন হিসেবে তালিকাভুক্ত করা হয়েছে তা উল্লেখ নেই।
NEW ANSWER: ২০১৯ সালে
QUESTION ID: test_0823
OLD ANSWER: তথ্য নেই
NEW ANSWER: Context-এ তথ্য নেই
QUESTION ID: test_1016
OLD ANSWER: তথ্য নেই
NEW ANSWER: ১০.২৬%
QUESTION ID: test_1414
OLD ANSWER: প্রদত্ত তথ্যে নেই
NEW ANSWER: Co

# Post Processing sub 16_1 (sub16_2)

In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


!pip install -q duckduckgo-search
!pip install -q beautifulsoup4
!pip install -q requests
!pip install -q ddgs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 47.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.4 MB/s eta 0:00:00


In [5]:
# ==========================================================
# SILENCE ANNOYING WARNINGS
# RUN THIS AS FIRST CELL
# ==========================================================

import os
import warnings

# suppress ALL warnings
warnings.simplefilter("ignore")

# specifically suppress deprecation spam
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow"
)

# silence python warnings globally
os.environ["PYTHONWARNINGS"] = "ignore"

In [4]:
# ==========================================================
# AUTO ENTITY -> DUAL RETRIEVAL QA
# (DIRECT WIKI vs DDG -> WIKI)
# MULTI-QID SUPPORT
# ==========================================================

!pip install -q wikipedia-api ddgs

# ==========================================================
# IMPORTS
# ==========================================================

import warnings
import logging
import pandas as pd
import wikipediaapi

from ddgs import DDGS

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)


# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv(
    "/kaggle/input/datasets/rathiaich/sub8-rag-test-questions/test_with_rag.csv"
)

print(df.shape)
print(df.columns)

print("\nSample QIDs:")
print(
    df["index"]
    .head()
    .tolist()
)


# ==========================================================
# ENTITY EXTRACTION
# ==========================================================

def extract_entity(question):

    prompt = f"""
You are an expert entity extractor.

Task:
Find ONLY the MAIN SUBJECT ENTITY
of the question.

Question:
{question}

VERY IMPORTANT RULES:

1. Always extract the PERSON /
ORGANIZATION / PLACE / THING
the question is ABOUT.

2. NEVER extract:
- answer
- location asked in question
- date
- event
- attribute

3. For "where" questions,
extract WHO the question is about,
NOT the location.

4. For "when" questions,
extract WHAT the question is about,
NOT the date.

5. Keep FULL official name.

Output format:
Line 1 = Bengali entity
Line 2 = English entity


Examples:

Question:
ঢাকা বিশ্ববিদ্যালয় কত সালে প্রতিষ্ঠিত হয়?

Answer:
ঢাকা বিশ্ববিদ্যালয়
Dhaka University


Question:
রবীন্দ্রনাথ ঠাকুর কোন পুরস্কার পেয়েছিলেন?

Answer:
রবীন্দ্রনাথ ঠাকুর
Rabindranath Tagore


Question:
জাতিসংঘের মহাসচিব কে?

Answer:
জাতিসংঘ
United Nations

Only entity names.

CRITICAL RULES:

1. ALWAYS return the COMPLETE FULL NAME.
2. NEVER shorten person names.

CORRECT:
কেন উইলিয়ামসন
সাকিব আল হাসান
রবীন্দ্রনাথ ঠাকুর
শেখ মুজিবুর রহমান

WRONG:
উইলিয়ামসন
সাকিব
রবীন্দ্রনাথ
মুজিব


"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content":
                (
                    "Extract the MAIN "
                    "SUBJECT entity only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        top_p=1,
        max_tokens=40
    )

    output = response[
        "choices"
    ][0]["message"]["content"]

    lines = [
        x.strip()
        for x in output.split("\n")
        if x.strip()
    ]

    bn_entity = (
        lines[0]
        if len(lines) > 0
        else ""
    )

    en_entity = (
        lines[1]
        if len(lines) > 1
        else bn_entity
    )

    return bn_entity, en_entity


# ==========================================================
# WIKIPEDIA
# ==========================================================

wiki_bn = wikipediaapi.Wikipedia(
    language="bn",
    user_agent="rag-agent"
)

wiki_en = wikipediaapi.Wikipedia(
    language="en",
    user_agent="rag-agent"
)


# ==========================================================
# DIRECT WIKI RETRIEVAL
# ==========================================================

def get_direct_wiki_context(
    bn_entity,
    en_entity,
    max_chars=15000
):

    bn_text = ""
    en_text = ""

    try:

        bn_page = wiki_bn.page(
            bn_entity
        )

        if bn_page.exists():

            bn_text = (
                bn_page.text
                [:max_chars]
            )

    except:
        pass

    try:

        en_page = wiki_en.page(
            en_entity
        )

        if en_page.exists():

            en_text = (
                en_page.text
                [:max_chars]
            )

    except:
        pass

    return bn_text, en_text


# ==========================================================
# DDG SEARCH
# ==========================================================

def search_wikipedia_ddg(entity):

    try:

        query = (
            f"{entity} wikipedia"
        )

        results = DDGS().text(
            query,
            max_results=10
        )

        for r in results:

            url = r.get(
                "href",
                ""
            )

            if (
                "wikipedia.org/wiki/"
                in url
            ):
                return url

    except:
        pass

    return None


# ==========================================================
# EXTRACT TITLE
# ==========================================================

def wiki_title_from_url(url):

    try:

        title = (
            url
            .split("/wiki/")[-1]
            .replace("_", " ")
        )

        return title

    except:
        return None


# ==========================================================
# DDG -> WIKI RETRIEVAL
# ==========================================================

def get_ddg_wiki_context(
    bn_entity,
    en_entity,
    max_chars=15000
):

    bn_text = ""
    en_text = ""

    # Bengali entity
    bn_url = (
        search_wikipedia_ddg(
            bn_entity
        )
    )

    if bn_url:

        try:

            title = (
                wiki_title_from_url(
                    bn_url
                )
            )

            page = wiki_bn.page(
                title
            )

            if not page.exists():

                page = wiki_en.page(
                    title
                )

            if page.exists():

                bn_text = (
                    page.text
                    [:max_chars]
                )

        except:
            pass


    # English entity
    en_url = (
        search_wikipedia_ddg(
            en_entity
        )
    )

    if en_url:

        try:

            title = (
                wiki_title_from_url(
                    en_url
                )
            )

            page = wiki_en.page(
                title
            )

            if page.exists():

                en_text = (
                    page.text
                    [:max_chars]
                )

        except:
            pass

    return bn_text, en_text


# ==========================================================
# GEMMA QA
# ==========================================================

def ask_gemma(
    question,
    context,
    max_tokens=80
):

    prompt = f"""
প্রশ্নের semantic meaning বুঝে
Context থেকে সবচেয়ে উপযুক্ত উত্তর বের করো।

Question:
{question}

Context:
{context}

গুরুত্বপূর্ণ:
- শুধুমাত্র answer লিখবে
- sentence না
- reasoning না
- explanation না
"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content":
                "Answer only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        top_p=1,
        max_tokens=max_tokens
    )

    output = response[
        "choices"
    ][0]["message"]["content"]

    bad_strings = [
        "<think>",
        "</think>",
        "Thinking:",
        "Answer:",
        "উত্তর:"
    ]

    for s in bad_strings:

        output = output.replace(
            s,
            ""
        )

    output = (
        output.strip()
        .split("\n")[0]
        .strip()
    )

    return output


# ==========================================================
# RUN SINGLE QID
# ==========================================================

def run_qid(qid):

    qid = str(qid).strip()

    if qid.isdigit():

        qid = (
            f"test_{int(qid):04d}"
        )

    elif (
        qid.startswith("test_")
        and
        len(qid.split("_")[-1]) < 4
    ):

        num = int(
            qid.split("_")[-1]
        )

        qid = (
            f"test_{num:04d}"
        )

    matches = df[
        df["index"]
        .astype(str)
        == qid
    ]

    if len(matches) == 0:

        print(
            f"QID {qid} not found"
        )
        return None

    row = matches.iloc[0]

    question = row[
        "question"
    ]

    print("=" * 80)
    print("QID:", qid)
    print("=" * 80)

    print("\nQUESTION:")
    print(question)

    # ENTITY
    bn_entity, en_entity = (
        extract_entity(
            question
        )
    )

    print("\n" + "=" * 80)
    print("ENTITY")
    print("=" * 80)

    print("BN:", bn_entity)
    print("EN:", en_entity)

    # DIRECT WIKI
    direct_bn, direct_en = (
        get_direct_wiki_context(
            bn_entity,
            en_entity
        )
    )

    direct_context = f"""
[BANGLA]
{direct_bn}

[ENGLISH]
{direct_en}
"""

    direct_answer = ask_gemma(
        question,
        direct_context
    )

    # DDG -> WIKI
    ddg_bn, ddg_en = (
        get_ddg_wiki_context(
            bn_entity,
            en_entity
        )
    )

    ddg_context = f"""
[BANGLA]
{ddg_bn}

[ENGLISH]
{ddg_en}
"""

    ddg_answer = ask_gemma(
        question,
        ddg_context
    )

    print("\n" + "=" * 80)
    print("DIRECT WIKI ANSWER")
    print("=" * 80)
    print(direct_answer)

    print("\n" + "=" * 80)
    print("DDG -> WIKI ANSWER")
    print("=" * 80)
    print(ddg_answer)

    return {
        "qid": qid,
        "question": question,
        "direct_answer": direct_answer,
        "ddg_answer": ddg_answer
    }


# ==========================================================
# RUN MULTIPLE QIDS
# ==========================================================

def run_multiple_qids(qids):

    results = []

    total = len(qids)

    for i, qid in enumerate(qids, 1):

        print("\n" + "#" * 100)
        print(
            f"[{i}/{total}] RUNNING: {qid}"
        )
        print("#" * 100)

        try:

            result = run_qid(qid)

            if result is not None:
                results.append(
                    result
                )

        except Exception as e:

            print(
                f"FAILED: {qid}"
            )
            print(e)

    return pd.DataFrame(
        results
    )



# ==========================================================
# LOAD SUBMISSION_16(1)
# ==========================================================

submission16_1 = pd.read_csv(
    "/kaggle/working/submission_16(1).csv"
)

print("Submission_16(1):", submission16_1.shape)

# ==========================================================
# FIND FAILED ROWS
# ==========================================================

failed_rows = submission16_1[
    submission16_1["answer"]
    .astype(str)
    .str.strip()
    .eq("Context-এ তথ্য নেই")
]

print(
    "\nRows needing DDG retrieval:",
    len(failed_rows)
)

qids = (
    failed_rows["index"]
    .astype(str)
    .tolist()
)

print("\nQIDs:")
print(qids)

# ==========================================================
# RUN DDG -> WIKI RETRIEVAL
# ==========================================================

results_df = run_multiple_qids(
    qids
)

print("\nResults:")
print(results_df)

# ==========================================================
# UPDATE ANSWERS
# ==========================================================

submission16_2 = submission16_1.copy()

for _, row in results_df.iterrows():

    qid = str(
        row["qid"]
    ).strip()

    answer = str(
        row["ddg_answer"]
    ).strip()

    if answer:

        submission16_2.loc[
            submission16_2["index"]
            .astype(str)
            == qid,
            "answer"
        ] = answer

# ==========================================================
# SAVE SUBMISSION_16(2)
# ==========================================================

submission16_2.to_csv(
    "/kaggle/working/submission_16(2).csv",
    index=False
)

print(
    "\n✅ Saved: /kaggle/working/submission_16(2).csv"
)

# ==========================================================
# SHOW CHANGES
# ==========================================================

changed = submission16_2.merge(
    submission16_1,
    on="index",
    suffixes=("_new", "_old")
)

changed = changed[
    changed["answer_new"]
    != changed["answer_old"]
]

print(
    "\nRows changed:",
    len(changed)
)

if len(changed) > 0:

    print("\nChanged rows:\n")

    print(
        changed[
            [
                "index",
                "answer_old",
                "answer_new"
            ]
        ]
        .to_string(index=False)
    )

submission16_2.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
(1500, 6)
Index(['index', 'question', 'context1', 'context2', 'confidence1',
       'confidence2'],
      dtype='object')

Sample QIDs:
['test_0001', 'test_0002', 'test_0003', 'test_0004', 'test_0005']
Submission_16(1): (1500, 2)

Rows needing DDG retrieval: 7

QIDs:
['test_0350', 'test_0481

,index,answer
0,test_0001,শিক্ষাবিদ এবং মানব সম্পদ প্রশিক্ষক
1,test_0002,তার প্রাথমিক চরিত্রের কারণে
2,test_0003,স্মল মোজেস কোর্ট
3,test_0004,লেম্যান আর. ব্লেক
4,test_0005,পরাজিত দলের সদস্যদের ডাব্লিউডাব্লিউই হতে বরখাস...
